# Qubit Selector Workflow
IQM-Qubit-Selector is a client package which allows the user to pick optimal layouts for the user defined quantum circuit for the cystal topology IQM quantum hardware. This notebook demonstrates the workflow for using the IQM qubit-selector to optimize quantum circuit execution on IQM hardware. Note that currently this framework is based on ``qiskit`` and can be used for ``qiskit`` circuits.

**Key steps in the workflow:**

1. **Setup and Authentication**
    - Import required modules and set up your IQM token and iqm_server_url.

2. **Fetch Calibration Data (Optional)**
    - Retrieve calibration fidelities to visualize the it.

3. **Layout Generation and Cost Evaluation**
    - Use ``qubit_selector`` to generate possible qubit layouts for your circuit.
    - Evaluate the cost of each layout based on calibration data.

4. **Select the Best Layout**
    - Choose the layout with the lowest cost and transpile your circuit accordingly.

5. **Advanced Options**
    - Customize cost functions (e.g., include readout fidelity or use different two-qubit gate fidelities).
    - Remove specific qubits from consideration.
    - Generate additional layouts by increasing the number of trials.

Refer to the code cells for detailed usage and customization options.

In [12]:
import os
import matplotlib.pyplot as plt
from iqm.qubit_selector.qubit_selector import *
from iqm.qubit_selector.qiskit_utils import get_circuit, CircuitType
from iqm.qiskit_iqm import IQMProvider
from rustworkx.visualization import mpl_draw
from rustworkx import spring_layout

In [23]:
# Input your Resonance token
token = "XXXXXX"  # Replace with your actual token
os.environ["IQM_TOKEN"] = token

In [ ]:
iqm_server_url = "https://<your-iqm-server-url>" # Replace with your IQM server URL if needed
provider = IQMProvider(iqm_server_url)
backend = provider.get_backend()
os.environ["IQM_SERVER_URL"] = iqm_server_url ## needs to be initialized as an environment variable for qubit_selector.

### Fetch calibration data for the given device and visualize it 

In [15]:
calibration_data = CalibrationDataManager().get_calibration_fidelities(backend)

In [16]:
def plot_calibration_data(key, data):
    two_qubit_keys = [CalibrationType.CZ.value, CalibrationType.CLIFFORD.value]
    coherence_keys = [CalibrationType.T1.value, CalibrationType.T2.value]
    every_second = key in two_qubit_keys
    coherence = key in coherence_keys
    xlabel = 'Pairs' if key in two_qubit_keys else 'Qubits'
    ylabel = 'Error' if key not in coherence_keys else 'Coherence in (us)'
    pairs = list(data.items())
    if every_second:
        pairs = pairs[::2]
    labels = [pair[0] for pair in pairs]

    if coherence:
        values = [pair[1] for pair in pairs]
    else:
        values = [1-pair[1] for pair in pairs]

    plt.figure(figsize=(12, 6))
    plt.bar(labels, values)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f'{key} metrics')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

## uncomment for visualization of calibration data
# for k, v in calibration_data.items():
#     plot_calibration_data(k, v)

### Single API call to ``qubit_selector`` to get the best layout

Note that the lower the value of the cost, the better is the layout! The qubit indices in the layout are ``qiskit`` based.

In [17]:
nqubits = 10
# Possible choices for pre-defined circuits 'ghz', 'qft', 'random', 'wstate', 'quantum_volume' 'qaoa'
qc_algo = get_circuit(CircuitType.GHZ, nqubits) ## plug in your qiskit quantum circuit of interest

layouts, cost = CostEvaluator(backend=backend, quantum_circuit=qc_algo).get_top_layouts(num_layouts=50)
print("Top 10 qiskit layouts and their costs:")
for layout, c in zip(layouts[:10], cost[:10]):
    print(f"Layout: {layout}, Cost: {c*100:.2f}%")

[02-07 16:05:59;I] Number of layouts to evaluate: 1245
[02-07 16:06:14;I] Cost evaluation has begun using cost function "gate_cost_cz".
Top 10 qiskit layouts and their costs:
Layout: [0, 1, 4, 7, 8, 9, 10, 13, 14, 15], Cost: 4.32%
Layout: [0, 1, 4, 9, 10, 13, 15, 17, 18, 19], Cost: 4.40%
Layout: [1, 4, 8, 9, 13, 14, 15, 17, 18, 19], Cost: 4.45%
Layout: [0, 1, 4, 9, 10, 13, 14, 15, 17, 18], Cost: 4.50%
Layout: [0, 1, 4, 9, 13, 14, 15, 17, 18, 19], Cost: 4.58%
Layout: [4, 7, 8, 9, 10, 13, 15, 17, 18, 19], Cost: 4.59%
Layout: [7, 8, 9, 10, 13, 14, 15, 17, 18, 19], Cost: 4.69%
Layout: [0, 1, 4, 8, 9, 13, 15, 17, 18, 19], Cost: 4.72%
Layout: [4, 7, 8, 9, 13, 14, 15, 17, 18, 19], Cost: 4.75%
Layout: [0, 1, 4, 5, 7, 8, 10, 13, 14, 15], Cost: 4.76%


### Finding list of matching layouts to run your quantum circuit

In [18]:
layouts = LayoutGenerator(backend, qc_algo).generate_unique_layouts()

[02-07 16:06:14;I] Number of layouts to evaluate: 1245


### Adding readout fidelity in the cost function:

If you want to also add the readout error to the cost function, use the `readoutmode` argument in `CostEvaluator`. The `ReadoutMode.FIDELITY` option will add the readout fidelity to the cost function, while `ReadoutMode.QNDNESS` will add the readout QNDness metric. You can also choose to ignore the readout error by not providing the `readoutmode` argument or setting it to `ReadoutMode.NONE` (which is the default option).

In [19]:
layouts_with_readout, cost_with_readout = CostEvaluator(backend, qc_algo, readoutmode=ReadoutMode.QNDNESS).get_top_layouts(num_layouts=10)

[02-07 16:06:15;I] Number of layouts to evaluate: 1245
[02-07 16:06:31;I] Cost evaluation has begun using cost function "gate_cost_cz".


### Changing two-qubit gate fidelity in the cost function:

The `cost_function` parameter in the cost function allows you to specify which two-qubit gate fidelity that should be used when evaluating layouts. By default, the cost function uses the CZ gate fidelity (`CostFunction.GATE_COST_CZ`). If you want to use the Clifford gate fidelity instead, set `cost_function=CostFunction.GATE_COST_CLIFFORD` when calling `CostEvaluator`.

In [20]:
clifford_layouts, cost_clifford = CostEvaluator(backend, qc_algo, cost_function=CostFunction.GATE_COST_CLIFFORD,layouts= layouts, readoutmode=ReadoutMode.NONE).get_top_layouts(num_layouts=10)

[02-07 16:06:45;I] Cost evaluation has begun using cost function "gate_cost_clifford".


One can use additonal arguements to evalaute the layouts:
- Increase the number of layouts using `num_trials`. This is defaulted to `1000`.
    Note that this increases the run time of the cost-function evaluation.

In [21]:
more_layouts, cost = CostEvaluator(backend, qc_algo, num_trials=20000, readoutmode=ReadoutMode.NONE).get_top_layouts(num_layouts=10)

[02-07 16:06:46;I] Number of layouts to evaluate: 1320
[02-07 16:07:00;I] Cost evaluation has begun using cost function "gate_cost_cz".


### Removing default qubits:

There is an option to remove certain qubits prior to cost evalatuation of the layouts. So these qubits are not picked by the ``LayoutGenerator`` class.

In [22]:
qubits_to_remove = ["QB1", "QB2"]  ## plug in the qubits you want to remove from consideration
qubits_to_remove_qiskit_indices = [backend.qubit_name_to_index(qubit) for qubit in qubits_to_remove]  ## convert to indices assuming qubit names are in the format 'QB{index}'
layouts_with_removed_qubits, cost_with_removed_qubits = CostEvaluator(backend, qc_algo, remove_qubits=qubits_to_remove_qiskit_indices).get_top_layouts(num_layouts=10)

[02-07 16:07:01;I] Number of layouts to evaluate: 814
[02-07 16:07:11;I] Cost evaluation has begun using cost function "gate_cost_cz".
